In [1]:
%pip install nltk pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 26.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 33.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 39.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.2/807.2 kB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 38.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [nltk]2m11/12 [nltk]t-learn]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
import random
import pandas as pd
import numpy as np

from nltk.corpus import movie_reviews

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Download required NLTK data
nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


True

In [3]:
# Prepare movie reviews dataset
documents = [(list(movie_reviews.words(fileid)), category)
            for category in movie_reviews.categories()
            for fileid in movie_reviews.fileids(category)]

# Shuffle and prepare data
random.shuffle(documents)
reviews = [" ".join(words) for words, _ in documents]
sentiments = [sentiment for _, sentiment in documents]

# Create a DataFrame for better visualization
df = pd.DataFrame({
    'text': reviews,
    'sentiment': sentiments
})

print("Dataset Overview:")
print(df['sentiment'].value_counts())
print("\nSample Reviews:")
print(df.head())

Dataset Overview:
sentiment
pos    1000
neg    1000
Name: count, dtype: int64

Sample Reviews:
                                                text sentiment
0  note : some may consider portions of the follo...       pos
1  in 1977 , something never though possible happ...       pos
2  " deep rising " gives you that sinking feeling...       neg
3  i have always been a fan of director neil jord...       neg
4  seen february 15 , 1998 on home video ( borrow...       pos


In [4]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [5]:
# Create and configure CountVectorizer
count_vectorizer = CountVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
    stop_words='english'
)

# Transform text data
X_train_count = count_vectorizer.fit_transform(X_train)
X_test_count = count_vectorizer.transform(X_test)

# Display feature information
print("\nNumber of features:", len(count_vectorizer.get_feature_names_out()))
print("Top features:", count_vectorizer.get_feature_names_out()[:10])


Number of features: 5000
Top features: ['000' '10' '100' '11' '12' '13' '13th' '14' '15' '16']


In [8]:
# Create and configure TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
    stop_words='english'
)

# Transform text data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Display feature information
print("\nNumber of features:", len(tfidf_vectorizer.get_feature_names_out()))
print("Top features:", tfidf_vectorizer.get_feature_names_out()[:10])


Number of features: 5000
Top features: ['000' '10' '100' '11' '12' '13' '13th' '14' '15' '16']


In [9]:
# Initialize and train the classifier
tfidf_classifier = MultinomialNB()
tfidf_classifier.fit(X_train_tfidf, y_train)

# Make predictions
y_pred_tfidf = tfidf_classifier.predict(X_test_tfidf)

# Print detailed performance metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tfidf))

# Print overall accuracy
tfidf_accuracy = accuracy_score(y_test, y_pred_tfidf)
print(f"Accuracy: {tfidf_accuracy:.4f}")


Classification Report:
              precision    recall  f1-score   support

         neg       0.80      0.80      0.80       211
         pos       0.77      0.78      0.78       189

    accuracy                           0.79       400
   macro avg       0.79      0.79      0.79       400
weighted avg       0.79      0.79      0.79       400

Accuracy: 0.7875


In [11]:
# Initialize and train the SVM classifier
svm_classifier = LinearSVC(random_state=42, max_iter=2000)
svm_classifier.fit(X_train_count, y_train)

# Make predictions
y_pred_svm = svm_classifier.predict(X_test_count)

# Print detailed performance metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

# Print overall accuracy
svm_accuracy = accuracy_score(y_test, y_pred_svm)
print(f"Accuracy: {svm_accuracy:.4f}")


Classification Report:
              precision    recall  f1-score   support

         neg       0.85      0.82      0.83       211
         pos       0.80      0.84      0.82       189

    accuracy                           0.83       400
   macro avg       0.83      0.83      0.83       400
weighted avg       0.83      0.83      0.83       400

Accuracy: 0.8275


In [12]:
# Get feature importance scores
feature_names = count_vectorizer.get_feature_names_out()

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': abs(svm_classifier.coef_[0])
})

# Sort and display top important features
print("\nTop 10 Most Important Words for Classification:")
print(feature_importance.sort_values(
    'importance', ascending=False
).head(10))


Top 10 Most Important Words for Classification:
            feature  importance
4858          waste    0.337416
4954          words    0.173008
354           awful    0.159186
4968          worst    0.150905
4384       supposed    0.149684
2020          guess    0.144743
367             bad    0.141779
3198        overall    0.138022
4707  unfortunately    0.134519
1991          great    0.130022


### Explain the role of CountVectorizer in text classification. How does it prepare text data for machine learning? 2. What is the purpose of train_test_split? Why is it important in machine learning evaluation? 

CountVectorizer helps a computer understand text by converting words into numbers. It examines the frequency of each word to evaluate and classify sentiment—either positive or negative.

train_test_split is a method I use to train the computer, allowing me to determine whether it is producing accurate results or simply memorizing the data or making random guesses.

### Compare accuracy scores between the CountVectorizer and TfidfVectorizer - Document any differences in performance. Analyze why one method might perform better than the other 

CountVectorizer: used to determine the frequency of a word’s occurrence.

TfidfVectorizer: goes beyond simple word counting by also calculating the relative importance of a word within a review.

Typically, TfidfVectorizer tends to yield better results than CountVectorizer. However, in this PE04 assignment, CountVectorizer actually performed slightly better. This may be because, within this specific movie review dataset, simply counting the frequency of words was sufficient for the model to effectively identify sentiment. Adjusting for word importance via TF-IDF did not lead to an improvement in performance.

CountVectorizer: 79.25%
TfidfVectorizer: 78.75%

### Compare performance metrics between Naive Bayes and SVM. Discuss trade-offs between the two approaches - Consider training time, accuracy, and model complexity

Naive Bayes achieves an accuracy of 79.25%, while SVM reaches 82.75%. Thus, SVM yields better results. SVM also helps identify words that significantly influence classification, such as "waste," "awful," "worst," "bad," and "great." Naive Bayes is generally simpler and faster to train, whereas SVM can be more complex and time-consuming, though it sometimes offers higher accuracy.